# 02 · Calibración del K-Means: qué receta congela `configs/kmeans.json`

**Laboratorio en Python, no sistema.** Go no decide features, transformación, nivel ni k: los lee de
`configs/kmeans.json`. Este notebook corre la **grilla de la spec de calibración (§3)** con
scikit-learn sobre una submuestra sistemática del silver y aplica la **regla de elección** en su
orden, para proponer esa receta con sus cifras.

Grilla: features F1 {PM2.5, PM10} · F2 {PM10, fracción fina} × meteorología M0 · M1 (viento,
humedad) · M2 (M1 + temperatura, presión) × transformación T1 crudo · T2 log1p en contaminantes ×
nivel N1 absoluto · N2 por estación × k = 2…10 × 5 semillas (`KMeans(init="k-means++")`).

Por corrida: inercia, silueta (10 000 puntos), ARI entre semillas, ARI por remuestreo en bloques de
días, pureza y recall de la etiqueta ECA en los regímenes que exceden, NMI con estación, hora y mes,
tamaño del cluster más chico y distancia mínima entre centroides.

| Entra | Sale |
|---|---|
| `data/silver/oefa_ancho_prototipo.csv` (notebook 01) o `oefa_ancho.csv` (Go) | `reports/calibracion.csv` (una fila por corrida) y `calibracion_resumen.csv` |
| | `reports/calibracion_decision.md` (tabla de decisión) |
| | `reports/kmeans_propuesto.json` (mismo esquema que `configs/kmeans.json`) |
| | `reports/centroides_sklearn.json` (referencia para el test de equivalencia con Go) |
| §8: `reports/calibracion_resumen.csv`, silver y `data/gold/asignaciones.csv` (Go) | `reports/descriptores_regimen.csv` y la sección «Decisión del equipo» de `calibracion_decision.md` |

La propuesta **no** se copia sola a `configs/kmeans.json`: lo decide el equipo mirando la tabla. La
**§8, «Decisión del equipo»**, documenta la receta que se congeló (que no es la de la regla automática),
describe cada régimen del modelo de Go con la fracción fina y la meteorología, y deja
`kmeans_propuesto.json` y `centroides_sklearn.json` con esa receta. La §8 se corre sola, sin la grilla
(§0 y la celda de importaciones bastan): lee los CSV de `reports/` y las salidas de Go.

## 0. Entorno

En Colab basta con *Entorno de ejecución → Ejecutar todo*. Esta celda monta Drive (la primera vez en
cada sesión pide autorizarlo), deja el código en la rama con que se abrió el notebook, instala lo que
falte, imprime commit, raíz de los datos, CPU y RAM, y verifica que los insumos estén en
`MyDrive/aire-distribuido/`: si falta algo o está desfasado, dice qué subir y con qué comando de
`rclone`. Ver «Iterar en Colab» en `notebooks/README.md`.

In [ ]:
# ── 0. Entorno: la misma celda en los tres notebooks ──────────────────────────────────────────
# En Colab: monta Drive, deja el repo en /content/aire-distribuido en la rama con que se abrió el
# notebook (la de la URL; si no se puede leer, RAMA) con git fetch + reset --hard, instala lo que
# falte y dice qué commit, qué datos y qué máquina se usan. En local no toca git ni Drive: usa el
# repo donde está el notebook y la raíz de AIRE_RAIZ (o el repo).
RAMA = "feature/spec-pc3"   # rama del código si no se puede leer de la URL de Colab
ACTUALIZAR_CODIGO = True    # Colab: traer lo último de la rama en cada corrida (False: dejar el código como está)
RAIZ_DRIVE = "/content/drive/MyDrive/aire-distribuido"  # capas medallón en Drive
NOTEBOOK = "02"              # qué insumos se verifican al final de la celda

import importlib, importlib.util, json, os, re, subprocess, sys, time, urllib.parse, urllib.request
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
RAIZ_LOCAL = os.environ.get("AIRE_RAIZ")  # otra raíz de las capas (p. ej. el sintético); en Colab, sin Drive
REPO_GIT = "https://github.com/R0SEWT/aire-distribuido.git"
PAQUETES = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "scikit-learn": "sklearn",
            "scipy": "scipy", "joblib": "joblib"}  # Colab los trae: solo se instala lo que falte

if EN_COLAB:  # Colab corre en UTC: las fechas de reportes y figuras, en hora de Lima
    os.environ["TZ"] = "America/Lima"
    time.tzset()
if EN_COLAB and not RAIZ_LOCAL:
    from google.colab import drive
    # Cada entorno nuevo pide autorizar Drive; con «Ejecutar todo» se espera hasta 15 min (no los 2 por defecto).
    drive.mount("/content/drive", timeout_ms=15 * 60 * 1000)


def rama_de_colab():
    """La rama de la URL del notebook (colab…/github/R0SEWT/aire-distribuido/blob/<rama>/notebooks/…)."""
    for host in dict.fromkeys([os.environ.get("COLAB_JUPYTER_IP"), "172.28.0.12", "172.28.0.2", "127.0.0.1"]):
        if not host:
            continue
        try:
            puerto = os.environ.get("KMP_TARGET_PORT", "9000")
            with urllib.request.urlopen(f"http://{host}:{puerto}/api/sessions", timeout=3) as r:
                texto = urllib.parse.unquote(urllib.parse.unquote(r.read().decode()))
        except Exception:
            continue
        m = re.search(r"aire-distribuido/blob/(.+?)/notebooks/", texto)
        return m.group(1) if m else None
    return None


def git(*args, repo=None):
    r = subprocess.run(["git", *(["-C", str(repo)] if repo else []), *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)}: {r.stderr.strip()}")
    return r.stdout.strip()


def traer_repo(rama, destino=Path("/content/aire-distribuido")):
    """Clona la rama (la primera vez) o la pone al día con fetch + reset --hard (las siguientes)."""
    if not (destino / ".git").exists():
        subprocess.run(["rm", "-rf", str(destino)])
        git("clone", "-q", "--depth", "1", "-b", rama, REPO_GIT, str(destino))
    elif ACTUALIZAR_CODIGO:
        git("fetch", "-q", "--depth", "1", "origin", f"+refs/heads/{rama}:refs/remotes/origin/{rama}", repo=destino)
        git("checkout", "-q", "-f", "-B", rama, f"origin/{rama}", repo=destino)
        git("reset", "-q", "--hard", f"origin/{rama}", repo=destino)
    return destino


if EN_COLAB:
    RAMA_USADA, ORIGEN_RAMA = rama_de_colab(), "de la URL de Colab"
    if not RAMA_USADA:
        RAMA_USADA, ORIGEN_RAMA = RAMA, "de RAMA: no se pudo leer la URL"
    for rama in dict.fromkeys([RAMA_USADA, "develop"]):
        try:
            REPO = traer_repo(rama)
            break
        except RuntimeError as err:
            print(f"AVISO: no pude traer la rama {rama!r} ({err}); pruebo la siguiente.")
    else:
        raise RuntimeError(f"No pude clonar {REPO_GIT}: ¿hay internet en el entorno de ejecución?")
    if rama != RAMA_USADA:
        RAMA_USADA, ORIGEN_RAMA = rama, f"{RAMA_USADA!r} no existe en GitHub"
    if git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO) != RAMA_USADA:  # clon viejo sin actualizar
        RAMA_USADA, ORIGEN_RAMA = git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO), "clon anterior: ACTUALIZAR_CODIGO = False"
else:
    REPO = next((c for c in (Path.cwd(), Path.cwd().parent) if (c / "notebooks" / "aire_lab.py").exists()), None)
    if REPO is None:
        raise FileNotFoundError("No encuentro notebooks/aire_lab.py: abre el notebook desde notebooks/ o desde la raíz del repo.")
    RAMA_USADA, ORIGEN_RAMA = None, ""

faltan = [pip for pip, mod in PAQUETES.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

if str(REPO / "notebooks") not in sys.path:
    sys.path.insert(0, str(REPO / "notebooks"))
RAIZ = Path(RAIZ_LOCAL) if RAIZ_LOCAL else (Path(RAIZ_DRIVE) if EN_COLAB else REPO)
DATA, REPORTS = RAIZ / "data", RAIZ / "reports"
CONFIGS = [RAIZ / "configs", REPO / "configs"]  # la de la raíz (Drive), si existe, va primero
# Figuras del informe: en local, docs/informe/generado/figuras/; en Colab, reports/figuras/ de Drive.
FIGURAS = Path(os.environ["AIRE_FIGURAS"]) if os.environ.get("AIRE_FIGURAS") else (
    RAIZ / "reports" / "figuras" if EN_COLAB else REPO / "docs" / "informe" / "generado" / "figuras")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

import aire_lab as al
importlib.reload(al)
CMAPS = al.estilo_figuras()
SINTETICO = al.es_sintetico(RAIZ)  # se vuelve a mirar al leer los datos (estaciones SINT-/SIN-)
ENTORNO = al.describir_entorno(REPO, RAIZ, FIGURAS, EN_COLAB, RAMA_USADA, ORIGEN_RAMA)
INSUMOS = al.verificar_insumos(NOTEBOOK, RAIZ, CONFIGS, estricto=EN_COLAB)

In [ ]:
import json, math, time, itertools, warnings, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sklearn
from sklearn.cluster import KMeans

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
FIG_CAL = REPORTS / "figuras_calibracion"
FIG_CAL.mkdir(parents=True, exist_ok=True)


def guardar(fig, nombre):
    al.marca_sintetico(fig, SINTETICO)
    fig.savefig(FIG_CAL / nombre, dpi=150)
    plt.show()

## 1. Parámetros

Los umbrales son los de la spec (§3), puntos de partida que se reportan tal cual.
`K_DESDE_CODO` es la lectura de «el k más chico que cumpla lo anterior, **mirando el codo**»: el codo
de cada variante es el piso de k, para no elegir un k = 2 que parte el aire en «bajo» y «alto»
**[decisión mía, reversible]**.

In [ ]:
FUENTE_SILVER = None      # None: el silver de Go si existe, si no el prototipo del notebook 01; o una ruta
GRILLA_COMPLETA = None    # None: completa con más de 4 CPU (Colab con RAM alta, local); True/False la fuerza
N_CPU = al.cpus_disponibles()
if GRILLA_COMPLETA is None:
    GRILLA_COMPLETA = N_CPU > 4
# Completa: el silver entero (233 708 filas < 250 000) y silueta sobre 10 000 puntos; ~105 min de CPU.
# Reducida (2 CPU, Colab estándar): una de cada ~5 filas y silueta sobre 3 000 puntos.
N_SUBMUESTRA = 250_000 if GRILLA_COMPLETA else 50_000   # filas anchas de la submuestra; el paso se hace coprimo con 24
N_SILUETA = 10_000 if GRILLA_COMPLETA else 3_000        # la silueta es O(n²)
print(f"{N_CPU} CPU → grilla {'COMPLETA' if GRILLA_COMPLETA else 'REDUCIDA'}: N_SUBMUESTRA = {N_SUBMUESTRA:,}, "
      f"N_SILUETA = {N_SILUETA:,}" + ("" if GRILLA_COMPLETA else
      ". Las cifras cambian algo respecto de la completa; para la tabla del informe, GRILLA_COMPLETA = True "
      "(o Colab con RAM alta, que trae más CPU)."))
KS = list(range(2, 11))
SEMILLAS = [11, 23, 42, 57, 89]
N_JOBS = -1               # joblib: todos los núcleos
K_DESDE_CODO = True
GASES_EN_FEATURES = []    # p. ej. ["so2"] si el EDA dice que la gran mayoría de estaciones lo mide (spec §2.5)

UMBRALES = {
    "ari_semillas": 0.8,  # (1) estable
    "ari_dias": 0.7,
    "pureza": 0.6,        # (2) separa el ECA (etiqueta: promedio móvil de 24 h sobre el ECA)
    "recall": 0.5,
    "margen_geografia": 0.10,  # (3) NMI con estación <= la de una partición por nivel de PM10 + margen
    "min_cluster": 0.01,  # (4) interpretable: ningún cluster bajo el 1 %
    "sep_linf": 1.0,      #     y cada par de centroides difiere en > 1 desviación en alguna variable
}
# Lo que la grilla no calibra: se copia de configs/kmeans.json (Go los usa tal cual).
FIJOS = {"semilla": 42, "semillas": 10, "tolerancia": 1e-6, "max_iter": 100, "workers": 0, "chunk": 10000}
try:
    _actual, _ = al.leer_receta([c / "kmeans.json" for c in CONFIGS])
    FIJOS = {k: _actual.get(k, v) for k, v in FIJOS.items()}
except FileNotFoundError:
    pass

FAMILIA_F = {"F1": ["pm2_5", "pm10"], "F2": ["pm10", "fraccion_fina"]}
FAMILIA_M = {"M0": [], "M1": ["viento_velocidad", "humedad"],
             "M2": ["viento_velocidad", "humedad", "temperatura", "presion"]}
FAMILIA_T = {"T1": False, "T2": True}
FAMILIA_N = {"N1": "absoluto", "N2": "estacion"}

## 2. Silver y submuestra sistemática

In [ ]:
candidatos = [DATA / "silver" / "oefa_ancho.csv", DATA / "silver" / "oefa_ancho_prototipo.csv"]
RUTA_SILVER = Path(FUENTE_SILVER) if FUENTE_SILVER else next((c for c in candidatos if c.exists()), None)
if RUTA_SILVER is None or not RUTA_SILVER.exists():
    raise FileNotFoundError(f"No hay silver en {candidatos}: corre el notebook 01 o el cargador de Go.")
silver = al.leer_csv_contrato(RUTA_SILVER).sort_values(["estacion", "fecha_hora"], kind="mergesort").reset_index(drop=True)
SHA_SILVER = al.sha256(RUTA_SILVER)
SINTETICO = SINTETICO or al.es_sintetico(estaciones=silver["estacion"].unique())
# Etiqueta ECA de cada hora: el promedio móvil de 24 h (ventana que termina en la hora, >= 18 h con
# dato) supera 50 µg/m³ de PM2.5 o 100 de PM10. El ECA es de 24 h: la lectura horaria no lo es.
mv = al.movil_24h(al.regularizar(silver, [c for c in ["pm2_5", "pm10"] if c in silver]))
silver = silver.merge(mv[["estacion", "fecha_hora", "excede_24h"]], on=["estacion", "fecha_hora"], how="left")
silver["excede_24h"] = silver["excede_24h"].fillna(False).astype(bool)
# Paso coprimo con 24: un paso que divide al día (2, 3, 4, 6...) dejaría fuera horas fijas del día.
paso = max(1, math.ceil(len(silver) / N_SUBMUESTRA))
while math.gcd(paso, 24) != 1:
    paso += 1
sub = silver.iloc[::paso].reset_index(drop=True)
print(f"Silver: {RUTA_SILVER.name} · {len(silver):,} filas anchas · {silver['estacion'].nunique()} estaciones")
print(f"Submuestra sistemática: cada {paso}.ª fila (paso coprimo con 24) → {len(sub):,} filas")

# Metadatos de cada fila de la submuestra: no son features, sirven para medir.
cod_est = {e: i for i, e in enumerate(sorted(sub["estacion"].unique()))}
META = pd.DataFrame({
    "est": sub["estacion"].map(cod_est).to_numpy(),
    "hora": sub["fecha_hora"].dt.hour.to_numpy(),
    "mes": sub["fecha_hora"].dt.month.to_numpy(),
    "dia": (sub["fecha_hora"].dt.normalize() - pd.Timestamp("2000-01-01")).dt.days.to_numpy(),
    "eca": sub["excede_24h"].to_numpy(),
    "pm2_5": sub["pm2_5"].to_numpy() if "pm2_5" in sub else np.nan,
    "pm10": sub["pm10"].to_numpy() if "pm10" in sub else np.nan,
})
print(f"Horas con el promedio de 24 h sobre el ECA en la submuestra: {100 * META['eca'].mean():.2f} %")

## 3. Las 24 variantes

Cada variante usa sus filas completas (regla 8): M2 pierde las estaciones sin presión, F1 y F2 las
que no miden las dos PM. La tabla dice cuántas filas y estaciones quedan en cada una; a igualdad de
lo demás se prefiere la que cubre más estaciones.

In [ ]:
VARIANTES = {}
for f, m, t, n in itertools.product(FAMILIA_F, FAMILIA_M, FAMILIA_T, FAMILIA_N):
    feats = FAMILIA_F[f] + GASES_EN_FEATURES + FAMILIA_M[m]
    logs = [x for x in feats if x in al.CONTAMINANTES] if FAMILIA_T[t] else []
    receta = {"features": feats, "log1p": logs, "nivel": FAMILIA_N[n]}
    mascara = al.completas(sub, feats, logs).to_numpy()
    nombre = f"{f}-{m}-{t}-{n}"
    if mascara.sum() < 50 * max(KS):
        print(f"{nombre}: solo {mascara.sum()} filas completas, se omite")
        continue
    gold_v, stats_v = al.construir_gold(sub.loc[mascara].reset_index(drop=True), receta)
    VARIANTES[nombre] = {"F": f, "M": m, "T": t, "N": n, "receta": receta, "mascara": mascara,
                         "X": np.ascontiguousarray(gold_v[feats].to_numpy(dtype=float))}
tabla_var = pd.DataFrame([{"variante": k, "features": " + ".join(v["receta"]["features"]),
                           "log1p": " + ".join(v["receta"]["log1p"]) or "—", "nivel": v["receta"]["nivel"],
                           "filas": int(v["mascara"].sum()),
                           "estaciones": int(META.loc[v["mascara"], "est"].nunique())}
                          for k, v in VARIANTES.items()])
display(tabla_var)

## 4. La grilla

Una tarea por (variante, k) con sus cinco semillas: así el ARI entre semillas se calcula dentro de
la tarea. El remuestreo es por **bloques de días** (días con reemplazo, con todas sus horas), porque
las horas consecutivas no son independientes: ajustar sobre el remuestreo y predecir la submuestra
completa da el ARI por días. Un régimen «excede el ECA» si su PM2.5 medio supera 50 µg/m³ o su PM10
medio supera 100 µg/m³ (spec PC3, §6.3); pureza y recall se miden contra la etiqueta del **promedio
móvil de 24 h** (`aire_lab.movil_24h`: ventana que termina en la hora, al menos 18 h con dato), que
es lo que el ECA mide. Además se guarda el **mejor régimen** frente al ECA (el de mayor proporción de
horas sobre el ECA), aunque su media no lo supere: dice si alguna partición aísla los episodios.

In [ ]:
def evaluar(nombre, k, X, est, hora, mes, dia, eca, pm25, pm10, semillas, n_silueta):
    """Corre las semillas de una (variante, k) y devuelve una fila por corrida."""
    import time as _t
    import warnings as _w
    import numpy as _np
    from sklearn.cluster import KMeans as _KM
    from sklearn.metrics import adjusted_rand_score as _ari, normalized_mutual_info_score as _nmi, silhouette_score as _sil
    _w.filterwarnings("ignore")
    n = len(X)
    # Línea base de «geografía»: una partición en k niveles de PM10 (cuantiles) también lleva
    # información de la estación, porque las estaciones difieren en nivel. Un régimen es geografía si
    # sabe mucho más de la estación que ese corte por nivel.
    cortes = _np.unique(_np.nanquantile(pm10, _np.linspace(0, 1, k + 1)[1:-1]))
    nmi_base = float(_nmi(_np.digitize(pm10, cortes), est))
    orden = _np.argsort(dia, kind="stable")
    _, ini = _np.unique(dia[orden], return_index=True)
    bloques = _np.split(orden, ini[1:])
    filas, etiquetas = [], []
    for s in semillas:
        t0 = _t.perf_counter()
        km = _KM(n_clusters=k, init="k-means++", n_init=1, random_state=s).fit(X)
        lab = km.labels_
        C = km.cluster_centers_
        sil = float(_sil(X, lab, sample_size=min(n_silueta, n), random_state=0)) if len(_np.unique(lab)) > 1 else _np.nan
        rng = _np.random.default_rng(s)
        idx = _np.concatenate([bloques[i] for i in rng.integers(0, len(bloques), len(bloques))])
        km_b = _KM(n_clusters=k, init="k-means++", n_init=1, random_state=s).fit(X[idx])
        ari_d = float(_ari(lab, km_b.predict(X)))
        medias = {}
        for nom, v in (("pm2_5", pm25), ("pm10", pm10)):
            ok = _np.isfinite(v)
            cnt = _np.bincount(lab[ok], minlength=k)
            sm = _np.bincount(lab[ok], weights=v[ok], minlength=k)
            medias[nom] = _np.where(cnt > 0, sm / _np.maximum(cnt, 1), _np.nan)
        excede = (_np.nan_to_num(medias["pm2_5"]) > 50) | (_np.nan_to_num(medias["pm10"]) > 100)
        sel = excede[lab]
        p_eca_c = _np.bincount(lab, weights=eca.astype(float), minlength=k) / _np.maximum(_np.bincount(lab, minlength=k), 1)
        mejor_c = int(_np.argmax(p_eca_c))
        tam = _np.bincount(lab, minlength=k) / n
        dif = C[:, None, :] - C[None, :, :]
        tri = _np.triu_indices(k, 1)
        filas.append({
            "variante": nombre, "k": k, "semilla": s, "n_filas": n, "n_estaciones": int(len(_np.unique(est))),
            "inercia": float(km.inertia_), "iteraciones": int(km.n_iter_), "silueta": sil, "ari_dias": ari_d,
            "pureza": float(eca[sel].mean()) if sel.any() else _np.nan,
            "recall": float(sel[eca].mean()) if eca.any() else _np.nan,
            "frac_excede": float(sel.mean()), "n_regimenes_excede": int(excede.sum()),
            "enriquecimiento_max": float(p_eca_c.max() / eca.mean()) if eca.any() else _np.nan,
            "pureza_mejor": float(p_eca_c[mejor_c]),
            "recall_mejor": float((lab[eca] == mejor_c).mean()) if eca.any() else _np.nan,
            "frac_mejor": float((lab == mejor_c).mean()),
            "nmi_estacion_base": nmi_base,
            "nmi_estacion": float(_nmi(lab, est)), "nmi_hora": float(_nmi(lab, hora)),
            "nmi_mes": float(_nmi(lab, mes)), "nmi_eca": float(_nmi(lab, eca)),
            "min_cluster_frac": float(tam.min()),
            "dist_min_centroides": float(_np.sqrt((dif ** 2).sum(-1))[tri].min()),
            "sep_linf": float(_np.abs(dif).max(-1)[tri].min()),
            "segundos": _t.perf_counter() - t0,
        })
        etiquetas.append(lab)
    for i, f in enumerate(filas):
        f["ari_semillas"] = float(_np.mean([_ari(etiquetas[i], etiquetas[j]) for j in range(len(etiquetas)) if j != i]))
    return filas


tareas = []
for nombre, V in VARIANTES.items():
    mm = META[V["mascara"]]
    args = (V["X"], mm["est"].to_numpy(), mm["hora"].to_numpy(), mm["mes"].to_numpy(), mm["dia"].to_numpy(),
            mm["eca"].to_numpy(), mm["pm2_5"].to_numpy(dtype=float), mm["pm10"].to_numpy(dtype=float))
    for k in sorted(KS, reverse=True):  # los k grandes primero: reparten mejor la carga
        tareas.append((nombre, k) + args)
print(f"{len(tareas)} tareas × {len(SEMILLAS)} semillas = {len(tareas) * len(SEMILLAS)} corridas")

t0 = time.perf_counter()
try:
    from joblib import Parallel, delayed
    salidas = Parallel(n_jobs=N_JOBS, verbose=2)(delayed(evaluar)(*t, SEMILLAS, N_SILUETA) for t in tareas)
except ImportError:
    print("Sin joblib: la grilla corre en serie")
    salidas = [evaluar(*t, SEMILLAS, N_SILUETA) for t in tareas]
print(f"Grilla en {(time.perf_counter() - t0) / 60:.1f} min")

cal = pd.DataFrame([f for s in salidas for f in s])
for col in ["F", "M", "T", "N"]:
    cal[col] = cal["variante"].map({k: v[col] for k, v in VARIANTES.items()})
cal["features"] = cal["variante"].map({k: "+".join(v["receta"]["features"]) for k, v in VARIANTES.items()})
cal["log1p"] = cal["variante"].map({k: "+".join(v["receta"]["log1p"]) for k, v in VARIANTES.items()})
cal["nivel"] = cal["variante"].map({k: v["receta"]["nivel"] for k, v in VARIANTES.items()})
cal = cal.sort_values(["variante", "k", "semilla"]).reset_index(drop=True)
cal.to_csv(REPORTS / "calibracion.csv", index=False)
print("Escrito", REPORTS / "calibracion.csv", cal.shape)

## 5. Regla de elección

Promedio de las cinco semillas por (variante, k), y la regla de la spec **en este orden**:
(1) estable: ARI entre semillas > 0,8 y por días > 0,7; (2) separa el ECA: pureza > 0,6 y recall
> 0,5 en los regímenes que exceden (etiqueta de 24 h); (3) no es geografía: la NMI del régimen con
la estación no supera en más de 0,10 a la de una **línea base**, la partición de las mismas filas
en k niveles de PM10 (cuantiles) **[decisión mía, de la revisión]**. La regla de la spec (NMI con
estación < NMI con la etiqueta ECA) casi no puede cumplirse: la NMI con una etiqueta binaria y rara
tiene un techo bajo. Si el absoluto falla aquí, gana el nivel por estación; (4) interpretable: ningún cluster bajo
el 1 % y centroides distinguibles; (5) el k más chico que cumpla, desde el codo de la variante.
Un criterio que no cumple **nadie** se dice y se salta, y se siguen aplicando los demás en orden. Desempates a igual k: más estaciones cubiertas, mejor pureza × recall,
mejor silueta.

In [ ]:
res = (cal.assign(pureza0=cal["pureza"].fillna(0), recall0=cal["recall"].fillna(0))
       .groupby(["variante", "F", "M", "T", "N", "k"], as_index=False)
       .agg(n_filas=("n_filas", "first"), n_estaciones=("n_estaciones", "first"),
            inercia=("inercia", "mean"), silueta=("silueta", "mean"),
            ari_semillas=("ari_semillas", "mean"), ari_dias=("ari_dias", "mean"),
            pureza=("pureza0", "mean"), recall=("recall0", "mean"), frac_excede=("frac_excede", "mean"),
            nmi_estacion=("nmi_estacion", "mean"), nmi_hora=("nmi_hora", "mean"), nmi_mes=("nmi_mes", "mean"),
            nmi_eca=("nmi_eca", "mean"), nmi_estacion_base=("nmi_estacion_base", "mean"),
            pureza_mejor=("pureza_mejor", "mean"), recall_mejor=("recall_mejor", "mean"), frac_mejor=("frac_mejor", "mean"),
            min_cluster=("min_cluster_frac", "min"),
            sep_linf=("sep_linf", "mean"), dist_min=("dist_min_centroides", "mean")))
codos = res.groupby("variante").apply(lambda g: al.codo(g["k"], g["inercia"]), include_groups=False).rename("codo")
res = res.merge(codos, on="variante")
res["f_eca"] = res["pureza"] * res["recall"]
res["c1_estable"] = (res["ari_semillas"] > UMBRALES["ari_semillas"]) & (res["ari_dias"] > UMBRALES["ari_dias"])
res["c2_eca"] = (res["pureza"] > UMBRALES["pureza"]) & (res["recall"] > UMBRALES["recall"])
res["c3_no_geografia"] = res["nmi_estacion"] <= res["nmi_estacion_base"] + UMBRALES["margen_geografia"]
res["c4_interpretable"] = (res["min_cluster"] >= UMBRALES["min_cluster"]) & (res["sep_linf"] > UMBRALES["sep_linf"])
res["k_desde_codo"] = res["k"] >= res["codo"] if K_DESDE_CODO else True
CRIT = ["c1_estable", "c2_eca", "c3_no_geografia", "c4_interpretable"]
NOMBRE_C = {"c1_estable": "(1) estable", "c2_eca": "(2) separa el ECA", "c3_no_geografia": "(3) no es geografía",
            "c4_interpretable": "(4) interpretable"}
# Los criterios se aplican en orden; uno que no cumple NADIE (de los que quedan) se dice y se salta,
# y se sigue con los siguientes. «prefijo» cuenta cuántos de los criterios aplicados cumple cada fila.
pool = res[res["k_desde_codo"]]
APLICADOS, SALTADOS, filtro = [], [], pool
for c in CRIT:
    if filtro[c].any():
        filtro = filtro[filtro[c]]
        APLICADOS.append(c)
    else:
        SALTADOS.append(c)
res["prefijo"] = 0
for i in range(len(APLICADOS)):
    res.loc[res[APLICADOS[: i + 1]].all(axis=1), "prefijo"] = i + 1
res.to_csv(REPORTS / "calibracion_resumen.csv", index=False)

ORDEN = ["k", "n_estaciones", "f_eca", "silueta"]
ASC = [True, False, False, False]
pool = res[res["k_desde_codo"]]
alcanzado = len(APLICADOS)
cand = pool[pool["prefijo"] == alcanzado].sort_values(ORDEN, ascending=ASC)
elegida = cand.iloc[0]
otras = cand[cand["variante"] != elegida["variante"]]
if len(otras):
    segunda = otras.iloc[0]
else:
    resto = pool[pool["variante"] != elegida["variante"]].sort_values(["prefijo"] + ORDEN, ascending=[False] + ASC)
    segunda = resto.iloc[0] if len(resto) else None
prov = res[(res["variante"] == "F2-M1-T2-N1") & (res["k"] == 5)]
provisional = prov.iloc[0] if len(prov) else None

encadenados, quedan = [], pool
for c in CRIT:
    if c in APLICADOS:
        quedan = quedan[quedan[c]]
        encadenados.append(int(len(quedan)))
    else:
        encadenados.append("nadie: se salta")
cuantos = pd.DataFrame({
    "criterio": CRIT,
    "cumplen solos": [int(pool[c].sum()) for c in CRIT],
    "cumplen este y los anteriores aplicados": encadenados,
    "de": len(pool),
})
display(cuantos)
print(f"Elegida: {elegida['variante']} con k = {int(elegida['k'])} (cumple {', '.join(APLICADOS) or 'ninguno'}; "
      f"nadie cumple {', '.join(SALTADOS) or '—'})")
if segunda is not None:
    print(f"Segunda: {segunda['variante']} con k = {int(segunda['k'])}")
cols_ver = ["variante", "k", "codo", "n_estaciones", "ari_semillas", "ari_dias", "pureza", "recall", "nmi_estacion",
            "nmi_eca", "min_cluster", "sep_linf", "silueta", "prefijo"]
display(cand[cols_ver].head(10).round(3))

In [ ]:
# Mapa de la grilla: cuántos criterios cumple, en orden, cada (variante, k).
piv = res.pivot(index="variante", columns="k", values="prefijo").reindex(sorted(VARIANTES))
fig, ax = plt.subplots(figsize=(8.5, 0.32 * len(piv) + 1.6))
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap(["#f0efec", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab"])
im = ax.imshow(piv.to_numpy(), cmap=cmap, norm=BoundaryNorm(np.arange(-0.5, 5.5), 5), aspect="auto")
ax.set_xticks(range(len(piv.columns)), piv.columns)
ax.set_yticks(range(len(piv.index)), piv.index, fontsize=8)
ax.grid(False)
for i, var in enumerate(piv.index):
    c = codos.get(var)
    if c in list(piv.columns):
        ax.plot(list(piv.columns).index(c), i, marker="|", color=al.TINTA, markersize=10)
fi, fj = list(piv.index).index(elegida["variante"]), list(piv.columns).index(int(elegida["k"]))
ax.add_patch(plt.Rectangle((fj - 0.5, fi - 0.5), 1, 1, fill=False, edgecolor=al.CRITICO, linewidth=2))
cb = fig.colorbar(im, ax=ax, ticks=range(5), shrink=0.7)
cb.set_label("criterios aplicados que cumple, en orden (" + ", ".join(NOMBRE_C.get(c, c) for c in APLICADOS) + ")", fontsize=8)
ax.set_xlabel("k   (| = codo de la variante; recuadro = elegida)")
ax.set_title("Calibración: cuántos criterios cumple cada variante y k", loc="left")
guardar(fig, "calibracion_grilla.png")

g = res[res["variante"] == elegida["variante"]].sort_values("k")
fig, axs = plt.subplots(1, 3, figsize=(12, 3.3))
axs[0].plot(g["k"], g["inercia"], marker="o", color=al.PALETA[0])
axs[0].set_title("Inercia (media de 5 semillas)", loc="left")
axs[1].plot(g["k"], g["silueta"], marker="o", color=al.PALETA[0])
axs[1].set_title("Silueta", loc="left")
axs[2].plot(g["k"], g["ari_semillas"], marker="o", color=al.PALETA[0], label="ARI entre semillas")
axs[2].plot(g["k"], g["ari_dias"], marker="o", color=al.PALETA[1], label="ARI por días")
axs[2].axhline(UMBRALES["ari_dias"], color=al.TINTA_2, linewidth=0.8)
axs[2].set_title("Estabilidad", loc="left")
axs[2].legend()
for ax in axs:
    ax.axvline(elegida["k"], color=al.CRITICO, linewidth=1, linestyle="--")
    ax.set_xlabel("k")
fig.suptitle(f"Variante elegida {elegida['variante']}", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
guardar(fig, "calibracion_elegida.png")

## 6. Receta propuesta y centroides de referencia

La receta elegida se ajusta sobre **todo** el silver (no la submuestra), con las cinco semillas, y se
guarda la de menor inercia. Los centroides se ordenan por la primera feature (ascendente) para que
el orden no dependa de la numeración de scikit-learn; el test de Go los compara con asignación
húngara y tolerancia.

In [ ]:
receta = dict(VARIANTES[elegida["variante"]]["receta"])
propuesta = {"features": receta["features"], "log1p": receta["log1p"], "nivel": receta["nivel"],
             "k": int(elegida["k"]), **FIJOS}
(REPORTS / "kmeans_propuesto.json").write_text(json.dumps(propuesta, indent=2) + "\n", encoding="utf-8")
print(json.dumps(propuesta, indent=2))

gold_full, stats_full = al.construir_gold(silver, receta)
Xf = gold_full[receta["features"]].to_numpy(dtype=float)
mejor = None
for s in SEMILLAS:
    km = KMeans(n_clusters=propuesta["k"], init="k-means++", n_init=1, random_state=s).fit(Xf)
    if mejor is None or km.inertia_ < mejor[1].inertia_:
        mejor = (s, km)
s_mejor, km = mejor
orden_c = np.lexsort(km.cluster_centers_.T[::-1])
Cz = km.cluster_centers_[orden_c]
lab = np.argsort(orden_c)[km.labels_]
Co = al.desestandarizar(Cz, stats_full)
ref = {
    "receta": propuesta, "fuente_silver": RUTA_SILVER.name, "sha256_silver": SHA_SILVER,
    "n": int(len(Xf)), "media": stats_full["media"], "desv": stats_full["desv"],
    "sklearn": sklearn.__version__, "semilla_sklearn": int(s_mejor), "n_semillas": len(SEMILLAS),
    "inercia": float(km.inertia_), "iteraciones": int(km.n_iter_),
    "orden_centroides": "por la primera feature, ascendente",
    "tamanos": np.bincount(lab, minlength=propuesta["k"]).tolist(),
    "centroides_estandarizados": Cz.tolist(),
    "centroides_desestandarizados": None if Co is None else Co.tolist(),
    "origen": "sintetico" if SINTETICO else "oefa",
}
(REPORTS / "centroides_sklearn.json").write_text(json.dumps(ref, indent=2) + "\n", encoding="utf-8")
print(f"Centroides de referencia: {len(Xf):,} filas, inercia {km.inertia_:.4f}, semilla {s_mejor}")
display(pd.DataFrame(Cz, columns=receta["features"]).round(3))
if Co is not None:
    display(pd.DataFrame(Co, columns=receta["features"]).round(3))

## 7. `reports/calibracion_decision.md`

In [ ]:
NOMBRE_C = {"c1_estable": "(1) estable", "c2_eca": "(2) separa el ECA", "c3_no_geografia": "(3) no es geografía",
            "c4_interpretable": "(4) interpretable"}
METRICAS = [("k", "k", "{:.0f}"), ("codo", "codo de la variante", "{:.0f}"), ("n_filas", "filas (submuestra)", "{:,.0f}"),
            ("n_estaciones", "estaciones", "{:.0f}"), ("ari_semillas", "ARI entre semillas", "{:.3f}"),
            ("ari_dias", "ARI por días", "{:.3f}"), ("pureza", "pureza «excede»", "{:.3f}"), ("recall", "recall «excede»", "{:.3f}"),
            ("frac_excede", "fracción de horas en «excede»", "{:.3f}"), ("nmi_estacion", "NMI estación", "{:.3f}"),
            ("nmi_estacion_base", "NMI estación de la línea base (nivel de PM10)", "{:.3f}"),
            ("nmi_eca", "NMI etiqueta ECA (24 h)", "{:.3f}"), ("nmi_hora", "NMI hora", "{:.3f}"), ("nmi_mes", "NMI mes", "{:.3f}"),
            ("pureza_mejor", "pureza del mejor régimen frente al ECA", "{:.3f}"), ("recall_mejor", "recall del mejor régimen", "{:.3f}"),
            ("frac_mejor", "tamaño del mejor régimen", "{:.3f}"),
            ("min_cluster", "cluster más chico", "{:.3f}"), ("sep_linf", "separación L∞ mínima", "{:.2f}"),
            ("silueta", "silueta", "{:.3f}"), ("inercia", "inercia", "{:,.1f}")]


def col(r):
    if r is None:
        return ["—"] * (len(METRICAS) + len(CRIT) + 3)
    rec = VARIANTES[r["variante"]]["receta"]
    out = [f"`{r['variante']}`", ", ".join(rec["features"]), ", ".join(rec["log1p"]) or "ninguna"]
    out += ["sí" if r[c] else "no" for c in CRIT]
    out += [f.format(r[m]).replace(",", " ") for m, _, f in METRICAS]
    return out


filas_md = ["variante", "features", "log1p"] + [NOMBRE_C[c] for c in CRIT] + [n for _, n, _ in METRICAS]
tabla_dec = pd.DataFrame({"": filas_md, "elegida": col(elegida), "segunda": col(segunda),
                          "provisional (F2-M1-T2-N1, k = 5)": col(provisional)})

def por_que(r):
    partes = [f"cumple {', '.join(NOMBRE_C[c] for c in APLICADOS)}"
              + (f" (nadie cumple {', '.join(NOMBRE_C[c] for c in SALTADOS)}: se dice y se salta)" if SALTADOS else "")]
    partes.append("es el k más chico que lo logra" + (f" desde el codo (k = {int(r['codo'])})" if K_DESDE_CODO else ""))
    partes.append(f"cubre {int(r['n_estaciones'])} estaciones")
    partes.append(f"pureza × recall = {r['f_eca']:.3f}, silueta {r['silueta']:.3f}")
    return "; ".join(partes)

def cede(r, s):
    if s is None:
        return "no hay segunda variante con la que comparar."
    mejores = []
    for m, n, f in METRICAS[3:]:
        a, b = r[m], s[m]
        mayor_mejor = m != "nmi_estacion"
        if m in ("inercia", "frac_excede", "nmi_hora", "nmi_mes", "nmi_estacion_base", "frac_mejor"):  # descriptivas
            continue
        if (b > a + 1e-9) if mayor_mejor else (b < a - 1e-9):
            mejores.append(f"{n} ({f.format(b)} frente a {f.format(a)})")
    crit_s = [NOMBRE_C[c] for c in CRIT if s[c] and not r[c]]
    texto = (f"la segunda (`{s['variante']}`, k = {int(s['k'])}) es mejor en: " + ("; ".join(mejores) if mejores else "nada medible"))
    if crit_s:
        texto += f". Cumple además {', '.join(crit_s)}"
    razon = ("tiene un k mayor" if s["k"] > r["k"] else "cubre menos estaciones" if s["n_estaciones"] < r["n_estaciones"]
             else "tiene menor pureza × recall" if s["f_eca"] < r["f_eca"] else "tiene menor silueta")
    if s["prefijo"] < alcanzado:
        razon = f"cumple menos de los criterios aplicados ({int(s['prefijo'])} de {alcanzado})"
    return texto + f". Pierde porque {razon}."

nadie = [NOMBRE_C[c] + f" (umbral: {txt})" for c, txt in zip(CRIT, [
    f"ARI semillas > {UMBRALES['ari_semillas']} y por días > {UMBRALES['ari_dias']}",
    f"pureza > {UMBRALES['pureza']} y recall > {UMBRALES['recall']}",
    f"NMI estación <= NMI de la línea base por nivel de PM10 + {UMBRALES['margen_geografia']}",
    f"cluster más chico >= {UMBRALES['min_cluster']} y separación L∞ > {UMBRALES['sep_linf']}"]) if pool[c].sum() == 0]
cadena = (f"Como nadie cumple {', '.join(NOMBRE_C[c] for c in SALTADOS)}, ese criterio se salta y se aplican los demás en orden."
          if SALTADOS else "Al menos un candidato cumple los cuatro criterios.")

hoy = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
L = ["# Calibración del K-Means — tabla de decisión", "",
     f"> Generado por `notebooks/02_calibracion.ipynb` el {hoy}. Silver `{RUTA_SILVER.name}` (sha256 `{SHA_SILVER[:16]}…`), "
     f"{al.miles(len(silver))} filas anchas; submuestra sistemática cada {paso}.ª fila ({al.miles(len(sub))} filas). "
     f"{len(VARIANTES)} variantes × {len(KS)} valores de k × {len(SEMILLAS)} semillas = {len(cal)} corridas "
     f"(scikit-learn {sklearn.__version__}). Spec: `docs/specs/2026-10-31-kmeans-calibracion-design.md` §3."]
if SINTETICO:
    L += ["", "> **DATOS SINTÉTICOS.** La decisión de abajo prueba el notebook; no es la receta del proyecto."]
L += ["", "## Decisión", "", al.tabla_md(tabla_dec), "",
      f"**Variante elegida: `{elegida['variante']}` con k = {int(elegida['k'])}.** Por qué: {por_que(elegida)}.", "",
      f"**Qué cede frente a la segunda**: {cede(elegida, segunda)}", "",
      f"**Umbrales que no cumplió nadie** (desde el codo): " + ("; ".join(nadie) if nadie else "ninguno: cada criterio lo cumple al menos un candidato") + f". {cadena}", "",
      "Umbrales usados: " + ", ".join(f"{k} = {v}" for k, v in UMBRALES.items()) + f"; k desde el codo: {K_DESDE_CODO}.", "",
      "## Cuántos candidatos cumplen cada criterio", "", al.tabla_md(cuantos.assign(criterio=cuantos["criterio"].map(NOMBRE_C))), "",
      "## Mejor k de cada variante", "",
      "Para cada variante, el k más chico (desde su codo) con el prefijo de criterios más largo.", ""]
mejor_por_var = (pool.sort_values(["variante", "prefijo", "k"], ascending=[True, False, True]).groupby("variante").head(1)
                 .sort_values(["prefijo", "k", "n_estaciones"], ascending=[False, True, False]))
L += [al.tabla_md(mejor_por_var[["variante", "k", "codo", "prefijo", "n_estaciones", "ari_semillas", "ari_dias", "pureza",
                                 "recall", "nmi_estacion", "nmi_eca", "min_cluster", "sep_linf", "silueta"]].round(3), decimales=3), "",
      "## ¿Algún régimen aísla los episodios?", "",
      "Etiqueta: el promedio móvil de 24 h sobre el ECA (" + f"{100 * META['eca'].mean():.2f} % de las horas de la submuestra). "
      "«Excede» por la media del régimen: " + (f"{int((res['frac_excede'] > 0).sum())} de {len(res)} candidatos tienen algún régimen cuya "
      "media supera el ECA." if (res['frac_excede'] > 0).any() else "ningún candidato tiene un régimen cuya media supere el ECA.") +
      " El mejor régimen de cada candidato (el de mayor proporción de horas sobre el ECA), los diez de mayor pureza:", "",
      al.tabla_md(res.sort_values(["pureza_mejor", "recall_mejor"], ascending=False).head(10)[
          ["variante", "k", "pureza_mejor", "recall_mejor", "frac_mejor", "pureza", "recall", "frac_excede", "ari_semillas"]].round(3), decimales=3), "",
      "## Receta propuesta (`reports/kmeans_propuesto.json`)", "",
      "Mismo esquema que `configs/kmeans.json`. `semilla`, `tolerancia`, `max_iter`, `workers` y `chunk` no los calibra "
      "la grilla: se copian del `configs/kmeans.json` actual. Copiar la propuesta allí es decisión de Rody.", "",
      "```json", json.dumps(propuesta, indent=2), "```", "",
      f"Centroides de scikit-learn sobre todo el silver ({al.miles(len(Xf))} filas, semilla {s_mejor}, inercia {km.inertia_:.4f}) "
      "en `reports/centroides_sklearn.json`, para el test de equivalencia con Go.", ""]
(REPORTS / "calibracion_decision.md").write_text("\n".join(L), encoding="utf-8")
print("Escrito", REPORTS / "calibracion_decision.md")
display(Markdown("\n".join(L[:16])))

## 8. Decisión del equipo

La regla de la §5 es automática; la receta que lee Go la decide el equipo. El 01/11 se congeló en
`configs/kmeans.json` **`F1-M0-T1-N1` con k = 3** (PM2.5 y PM10 en µg/m³, sin log, nivel absoluto), no
la que eligió la regla (`F2-M0-T1-N2`, k = 4: PM10 y fracción fina estandarizadas por estación). Esta
sección lo deja escrito con las cifras de la grilla, describe cada régimen **del modelo de Go** con lo que
no entra al K-Means (fracción fina y meteorología) y deja `kmeans_propuesto.json` y
`centroides_sklearn.json` con la receta final.

**Se corre sola, sin la grilla**: basta con la celda de entorno (§0) y la de importaciones. Lee
`reports/calibracion_resumen.csv`, `reports/regimenes.json`, el silver de Go y `data/gold/asignaciones.csv`
(la salida de `cmd/entrenar`), y escribe `reports/descriptores_regimen.csv`, `kmeans_propuesto.json`,
`centroides_sklearn.json` y la sección «Decisión del equipo» al final de `reports/calibracion_decision.md`
(si ya estaba, la reemplaza; lo de arriba no se toca).

In [ ]:
# ── 8.1 La variante de la regla automática frente a la del equipo (desde reports/calibracion_resumen.csv) ──
AUTOMATICA = ("F2-M0-T1-N2", 4)   # la que eligió la regla de la §5
EQUIPO = ("F1-M0-T1-N1", 3)       # la que congeló el equipo en configs/kmeans.json
# Las mismas familias de la §1 (la sección no depende de que se haya corrido).
FAM_F = {"F1": ["pm2_5", "pm10"], "F2": ["pm10", "fraccion_fina"]}
FAM_M = {"M0": [], "M1": ["viento_velocidad", "humedad"], "M2": ["viento_velocidad", "humedad", "temperatura", "presion"]}
resumen_cal = pd.read_csv(REPORTS / "calibracion_resumen.csv")


def fila_cal(variante, k):
    f = resumen_cal[(resumen_cal["variante"] == variante) & (resumen_cal["k"] == k)]
    if len(f) != 1:
        raise ValueError(f"{variante} con k = {k} no está en reports/calibracion_resumen.csv")
    return f.iloc[0]


def receta_de(r):
    feats = FAM_F[r["F"]] + FAM_M[r["M"]]
    return {"features": feats, "log1p": [f for f in feats if f in al.CONTAMINANTES] if r["T"] == "T2" else [],
            "nivel": {"N1": "absoluto", "N2": "estacion"}[r["N"]], "k": int(r["k"])}


auto, equipo = fila_cal(*AUTOMATICA), fila_cal(*EQUIPO)
rec_auto, rec_equipo = receta_de(auto), receta_de(equipo)
receta_final, origen_final = al.leer_receta([c / "kmeans.json" for c in CONFIGS])
iguales = all(receta_final.get(c) == v for c, v in rec_equipo.items())
print(f"configs/kmeans.json {'=' if iguales else '≠'} {EQUIPO[0]} con k = {EQUIPO[1]}: {json.dumps(receta_final)}")
if not iguales:
    print("AVISO: configs/kmeans.json ya no es la receta del equipo; revisar EQUIPO antes de escribir la sección.")

si_no = lambda x: "sí" if bool(x) else "no"
NIVEL_TXT = {"absoluto": "absoluto", "estacion": "por estación"}
COMPARA = [  # (fila de la tabla, función de la fila del resumen)
    ("features", lambda r: ", ".join(al.NOMBRE_TXT.get(f, f) for f in receta_de(r)["features"])),
    ("log1p", lambda r: ", ".join(receta_de(r)["log1p"]) or "ninguna"),
    ("nivel", lambda r: NIVEL_TXT[receta_de(r)["nivel"]]),
    ("k", lambda r: f"{int(r['k'])}"),
    ("codo de la variante", lambda r: f"{int(r['codo'])}"),
    ("k desde el codo (piso de la regla)", lambda r: si_no(r["k_desde_codo"])),
    ("(1) estable", lambda r: si_no(r["c1_estable"])),
    ("(2) separa el ECA", lambda r: si_no(r["c2_eca"])),
    ("(3) no es geografía", lambda r: si_no(r["c3_no_geografia"])),
    ("(4) interpretable", lambda r: si_no(r["c4_interpretable"])),
    ("estaciones", lambda r: f"{int(r['n_estaciones'])}"),
    ("horas (filas completas)", lambda r: al.miles(r["n_filas"])),
    ("ARI entre semillas", lambda r: f"{r['ari_semillas']:.3f}"),
    ("ARI entre remuestreos por días", lambda r: f"{r['ari_dias']:.3f}"),
    ("NMI con la estación", lambda r: f"{r['nmi_estacion']:.3f}"),
    ("NMI de la línea base (k niveles de PM10)", lambda r: f"{r['nmi_estacion_base']:.3f}"),
    ("NMI con la hora", lambda r: f"{r['nmi_hora']:.3f}"),
    ("silueta", lambda r: f"{r['silueta']:.3f}"),
    ("horas en los regímenes que exceden el ECA", lambda r: f"{100 * r['frac_excede']:.1f} %"),
    ("pureza «excede» (24 h sobre el ECA)", lambda r: f"{r['pureza']:.3f}"),
    ("recall «excede» (24 h sobre el ECA)", lambda r: f"{r['recall']:.3f}"),
    ("cluster más chico", lambda r: f"{100 * r['min_cluster']:.1f} %"),
    ("separación L∞ mínima entre centroides", lambda r: f"{r['sep_linf']:.2f}"),
]
tabla_equipo = pd.DataFrame({"": ["variante"] + [n for n, _ in COMPARA],
                             "regla automática": [f"`{auto['variante']}`"] + [f(auto) for _, f in COMPARA],
                             "equipo (receta final)": [f"`{equipo['variante']}`"] + [f(equipo) for _, f in COMPARA]})
display(tabla_equipo)

# La silueta del equipo frente a la grilla: ¿quién la supera, y con qué k?
mejores_sil = resumen_cal[resumen_cal["silueta"] > equipo["silueta"]]
k3 = resumen_cal[resumen_cal["k"] >= 3]
es_mejor_k3 = bool(k3["silueta"].max() <= equipo["silueta"])
ks_mejores = sorted(set(mejores_sil["k"].astype(int)))
TXT_SILUETA = (f"la mejor de la grilla con k ≥ 3" + (f"; solo la superan {len(mejores_sil)} candidatos, todos con k = "
               f"{', '.join(map(str, ks_mejores))}" if len(mejores_sil) else "") if es_mejor_k3 else
               f"la superan {len(mejores_sil)} candidatos de la grilla")
print("Silueta del equipo:", round(float(equipo["silueta"]), 3), "→", TXT_SILUETA)

In [ ]:
# ── 8.2 Descriptores por régimen: lo que no entra al K-Means, sobre el modelo de Go ──────────────
# Las asignaciones de Go (data/gold/asignaciones.csv) unidas al silver de Go por estación y fecha-hora.
# La fracción fina (PM2.5/PM10 recortada a [0, 1]) y la meteorología se describen, no son features.
ruta_silver_go = DATA / "silver" / "oefa_ancho.csv"
ruta_asig = DATA / "gold" / "asignaciones.csv"
if not (ruta_silver_go.exists() and ruta_asig.exists() and (REPORTS / "regimenes.json").exists()):
    raise FileNotFoundError("Faltan salidas de Go: data/silver/oefa_ancho.csv, data/gold/asignaciones.csv o reports/regimenes.json")
silver_go = al.leer_csv_contrato(ruta_silver_go)
asig_go = al.leer_csv_contrato(ruta_asig)
reg_go = json.loads((REPORTS / "regimenes.json").read_text(encoding="utf-8"))
mv_go = al.movil_24h(al.regularizar(silver_go, ["pm2_5", "pm10"]))
dd = (asig_go.merge(silver_go, on=["estacion", "fecha_hora"], how="left", validate="1:1")
      .merge(mv_go[["estacion", "fecha_hora", "excede_24h"]], on=["estacion", "fecha_hora"], how="left"))
assert dd["pm10"].notna().all(), "hay asignaciones sin fila en el silver"
dd["fraccion_fina"] = al.fraccion_fina(dd)
dd["excede_24h"] = dd["excede_24h"].fillna(False).astype(bool)
DESCR = ["fraccion_fina", "viento_velocidad", "humedad", "temperatura"]
METEO_D = DESCR[1:]
# Anomalía dentro de la estación: el valor menos la media de su estación (sobre las horas del gold). Así la
# diferencia entre regímenes no es solo qué estaciones caen en cada uno.
for v in METEO_D:
    dd[v + "_anomalia"] = dd[v] - dd.groupby("estacion")[v].transform("mean")

por_reg = {r["regimen"]: r for r in reg_go["regimenes"]}
orden_pm = sorted(por_reg, key=lambda r: por_reg[r].get("pm10_medio") or 0)
nombres = dict(zip(orden_pm, {2: ["bajo", "alto"], 3: ["bajo", "medio", "alto"]}.get(len(orden_pm), [""] * len(orden_pm))))
total_exc = int(dd["excede_24h"].sum())
filas_d = []
for r in orden_pm:
    g = dd[dd["regimen"] == r]
    f = {"regimen": r, "nombre": nombres[r], "horas": len(g), "fraccion_horas": len(g) / len(dd),
         "estaciones": g["estacion"].nunique(), "pm2_5_medio": g["pm2_5"].mean(), "pm10_medio": g["pm10"].mean(),
         "eca": por_reg[r].get("eca"),
         "horas_24h_sobre_eca": float(g["excede_24h"].mean()),
         "parte_de_las_horas_24h_sobre_eca": float(g["excede_24h"].sum() / total_exc) if total_exc else float("nan")}
    for v in DESCR:
        f[v + "_media"] = g[v].mean()
        f[v + "_horas"] = int(g[v].notna().sum())
        f[v + "_estaciones"] = int(g.loc[g[v].notna(), "estacion"].nunique())
    for v in METEO_D:
        f[v + "_anomalia_media"] = g[v + "_anomalia"].mean()
    filas_d.append(f)
descriptores = pd.DataFrame(filas_d)
descriptores.to_csv(REPORTS / "descriptores_regimen.csv", index=False)
print("Escrito", REPORTS / "descriptores_regimen.csv", descriptores.shape)

# Control: las medias de PM del silver son las de regimenes.json (perfil de Go) y las horas, sus tamaños.
dif_pm = max(max(abs(f["pm2_5_medio"] - por_reg[f["regimen"]]["pm2_5_medio"]) / por_reg[f["regimen"]]["pm2_5_medio"],
                 abs(f["pm10_medio"] - por_reg[f["regimen"]]["pm10_medio"]) / por_reg[f["regimen"]]["pm10_medio"]) for f in filas_d)
print(f"PM medio frente a regimenes.json: diferencia relativa máxima {dif_pm:.1e}; tamaños iguales: "
      f"{all(f['horas'] == por_reg[f['regimen']]['tamano'] for f in filas_d)}")
n_est_gold = dd["estacion"].nunique()
n_est_meteo = {v: int(dd.loc[dd[v].notna(), "estacion"].nunique()) for v in DESCR}
print(f"Estaciones del gold: {n_est_gold}; con cada descriptor: {n_est_meteo}")
display(descriptores.round(3))

In [ ]:
# ── 8.3 kmeans_propuesto.json y centroides_sklearn.json con la receta final ───────────────────
# Lo que la grilla no calibra (semilla, semillas, tolerancia, max_iter, workers, chunk) sale de configs/kmeans.json.
FIJOS_FINAL = ["semilla", "semillas", "tolerancia", "max_iter", "workers", "chunk"]
propuesta_final = {**rec_equipo, **{c: receta_final[c] for c in FIJOS_FINAL if c in receta_final}}
(REPORTS / "kmeans_propuesto.json").write_text(json.dumps(propuesta_final, indent=2) + "\n", encoding="utf-8")
print("kmeans_propuesto.json", "= configs/kmeans.json" if propuesta_final == receta_final else "≠ configs/kmeans.json",
      json.dumps(propuesta_final))

# Centroides de referencia: scikit-learn sobre todo el silver de Go, la mejor de cinco semillas, hasta converger
# (tol = 0; con su tol por defecto, 1e-4 de la varianza, para antes que Go). Mismo esquema que la §6.
SEMILLAS_REF = globals().get("SEMILLAS", [11, 23, 42, 57, 89])
gold_eq, stats_eq = al.construir_gold(silver_go, propuesta_final)
X_eq = gold_eq[propuesta_final["features"]].to_numpy(dtype=float)
# La de menor inercia (calculada aquí, no con inertia_, que varía en el último bit con los hilos); si
# empatan a 1e-9, la semilla menor: así el archivo no cambia de una corrida a otra.
corridas_eq = []
for s in SEMILLAS_REF:
    km_s = KMeans(n_clusters=propuesta_final["k"], init="k-means++", n_init=1, tol=0, max_iter=1000, random_state=s).fit(X_eq)
    corridas_eq.append((float(al.asignar(X_eq, km_s.cluster_centers_)[1].sum()), s, km_s))
menor_eq = min(c[0] for c in corridas_eq)
_, s_eq, km_eq = min((c for c in corridas_eq if c[0] <= menor_eq * (1 + 1e-9)), key=lambda c: c[1])
orden_eq = np.lexsort(km_eq.cluster_centers_.T[::-1])
Cz_eq = km_eq.cluster_centers_[orden_eq]
lab_eq = np.argsort(orden_eq)[km_eq.labels_]
in_eq = float(al.asignar(X_eq, Cz_eq)[1].sum())
ref_eq = {
    "receta": propuesta_final, "fuente_silver": ruta_silver_go.name, "sha256_silver": al.sha256(ruta_silver_go),
    "n": int(len(X_eq)), "media": stats_eq["media"], "desv": stats_eq["desv"],
    "sklearn": sklearn.__version__, "semilla_sklearn": int(s_eq), "n_semillas": len(SEMILLAS_REF), "tol": 0,
    "inercia": in_eq, "iteraciones": int(km_eq.n_iter_),
    "orden_centroides": "por la primera feature, ascendente",
    "tamanos": np.bincount(lab_eq, minlength=propuesta_final["k"]).tolist(),
    "centroides_estandarizados": Cz_eq.tolist(),
    "centroides_desestandarizados": (lambda c: None if c is None else c.tolist())(al.desestandarizar(Cz_eq, stats_eq)),
    "origen": "sintetico" if SINTETICO else "oefa",
}
(REPORTS / "centroides_sklearn.json").write_text(json.dumps(ref_eq, indent=2) + "\n", encoding="utf-8")
# Frente a Go: mismos centroides tras emparejar etiquetas (asignación húngara).
km_go_json = json.loads((REPORTS / "kmeans.json").read_text(encoding="utf-8")) if (REPORTS / "kmeans.json").exists() else None
DIF_GO = None
if km_go_json is not None and km_go_json.get("receta", {}).get("features") == propuesta_final["features"]:
    Cg = np.asarray(km_go_json["centroides_estandarizados"], float)
    DIF_GO = (float(np.abs(Cz_eq[al.emparejar(Cg, Cz_eq)] - Cg).max()), abs(in_eq - km_go_json["inercia"]) / km_go_json["inercia"])
print(f"centroides_sklearn.json: {al.miles(len(X_eq))} filas, semilla {s_eq}, inercia {in_eq:.6f}"
      + (f"; frente a Go: máx |Δ centroide| {DIF_GO[0]:.1e}, inercia relativa {DIF_GO[1]:.1e}" if DIF_GO else ""))

In [ ]:
# ── 8.4 La sección «Decisión del equipo» al final de reports/calibracion_decision.md ────────────
UMBRALES_TXT = globals().get("UMBRALES", {"pureza": 0.6, "recall": 0.5})
f3 = lambda x: f"{x:.3f}"
tabla_desc = pd.DataFrame({
    "régimen": [f"R{f['regimen']} {f['nombre']}".strip() for f in filas_d],
    "horas": [f"{al.miles(f['horas'])} ({100 * f['fraccion_horas']:.1f} %)" for f in filas_d],
    "PM2.5 / PM10 medios (µg/m³)": [f"{f['pm2_5_medio']:.1f} / {f['pm10_medio']:.1f}" for f in filas_d],
    "ECA (media)": [str(f["eca"]).replace("_", " ") for f in filas_d],
    "horas con 24 h sobre el ECA": [f"{100 * f['horas_24h_sobre_eca']:.1f} %" for f in filas_d],
    "parte del total de esas horas": [f"{100 * f['parte_de_las_horas_24h_sobre_eca']:.1f} %" for f in filas_d],
    "fracción fina": [f"{f['fraccion_fina_media']:.3f}" for f in filas_d],
    "viento (m/s)": [f"{f['viento_velocidad_media']:.2f}" for f in filas_d],
    "humedad (%)": [f"{f['humedad_media']:.1f}" for f in filas_d],
    "temperatura (°C)": [f"{f['temperatura_media']:.1f}" for f in filas_d],
})
tabla_anom = pd.DataFrame({
    "régimen": [f"R{f['regimen']} {f['nombre']}".strip() for f in filas_d],
    "estaciones con meteorología": [f"{f['viento_velocidad_estaciones']} de {f['estaciones']}" for f in filas_d],
    "horas con viento / humedad / temperatura": [f"{al.miles(f['viento_velocidad_horas'])} / {al.miles(f['humedad_horas'])} / "
                                                 f"{al.miles(f['temperatura_horas'])}" for f in filas_d],
    "viento: anomalía (m/s)": [f"{f['viento_velocidad_anomalia_media']:+.2f}" for f in filas_d],
    "humedad: anomalía (puntos)": [f"{f['humedad_anomalia_media']:+.1f}" for f in filas_d],
    "temperatura: anomalía (°C)": [f"{f['temperatura_anomalia_media']:+.1f}" for f in filas_d],
})
ALTO = filas_d[-1]
hoy = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
S = ["## Decisión del equipo (receta congelada en `configs/kmeans.json`)", "",
     f"> Agregado por la §8 de `notebooks/02_calibracion.ipynb` el {hoy}, con las cifras de `reports/calibracion_resumen.csv` "
     "(la grilla de arriba, sin volver a correrla) y, para los descriptores, el modelo de Go (`data/gold/asignaciones.csv`, "
     "`reports/regimenes.json`) unido al silver de Go.", "",
     f"La regla automática eligió `{auto['variante']}` con k = {int(auto['k'])} ({', '.join(al.NOMBRE_TXT.get(f, f) for f in rec_auto['features'])}, "
     f"nivel {NIVEL_TXT[rec_auto['nivel']]}). El equipo congeló **`{equipo['variante']}` con k = {int(equipo['k'])}**: "
     f"{' y '.join(al.NOMBRE_TXT.get(f, f) for f in rec_equipo['features'])} en µg/m³, sin log, nivel {NIVEL_TXT[rec_equipo['nivel']]}, "
     f"{receta_final.get('semillas')} semillas y chunk {al.miles(receta_final.get('chunk', 0))} (`configs/kmeans.json`).", "",
     al.tabla_md(tabla_equipo), "",
     f"**Por qué `{equipo['variante']}` con k = {int(equipo['k'])}**:", "",
     "- **Un solo K-Means de tres niveles de aire, que se explica y se defiende.** Sus regímenes son niveles absolutos "
     f"({', '.join(nombres[r] for r in orden_pm)}) y el ECA se lee sobre ellos. La automática necesita dos capas: un régimen "
     "relativo a cada estación (nivel por estación) y la etiqueta ECA calculada aparte, porque un z-score por estación no "
     "son µg/m³.",
     f"- **Estable**: ARI {f3(equipo['ari_semillas'])} entre semillas y {f3(equipo['ari_dias'])} entre remuestreos por días.",
     f"- **No es geografía**: NMI con la estación {f3(equipo['nmi_estacion'])}, frente a {f3(equipo['nmi_estacion_base'])} "
     "de la línea base (la partición en k niveles de PM10).",
     f"- **Silueta {f3(equipo['silueta'])}**: {TXT_SILUETA}.",
     f"- **{int(equipo['n_estaciones'])} estaciones** ({al.miles(equipo['n_filas'])} horas completas).",
     f"- **Su régimen alto excede el ECA en promedio y recoge el {100 * equipo['recall']:.0f} % de las horas sobre el ECA** "
     f"(promedio móvil de 24 h; pureza {equipo['pureza']:.2f}, con el {100 * equipo['frac_excede']:.1f} % de las horas). "
     f"La automática: pureza {auto['pureza']:.2f} y recall {auto['recall']:.2f}.",
     "- **La fracción fina y la meteorología se describen por régimen, no son features** (tablas de abajo).", "",
     f"**Qué cede**: k = {int(equipo['k'])} está por debajo del codo de su variante (k = {int(equipo['codo'])}), y la regla "
     "automática toma el codo como piso de k: por eso no la consideró. Cumple los otros tres criterios aplicados "
     f"(estable, no es geografía, interpretable) y, como todas, no cumple (2) separa el ECA (pureza > {UMBRALES_TXT['pureza']} y "
     f"recall > {UMBRALES_TXT['recall']}). Su cluster más chico tiene el {100 * equipo['min_cluster']:.1f} % de las horas y la "
     f"separación L∞ mínima entre centroides es {equipo['sep_linf']:.2f} desviaciones.", "",
     "### Descriptores por régimen (modelo de Go)", "",
     f"Medias sobre las horas del gold de cada régimen ({al.miles(len(dd))} horas, {n_est_gold} estaciones). Horas con "
     "24 h sobre el ECA: el promedio móvil de 24 h de PM2.5 o PM10 (ventana que termina en la hora, al menos 18 h con dato) "
     "supera 50 o 100 µg/m³; «parte del total» es qué proporción de todas esas horas cae en el régimen. "
     f"En `reports/descriptores_regimen.csv`, con cuántas horas y estaciones entran en cada media.", "",
     al.tabla_md(tabla_desc), "",
     f"La meteorología solo existe en {n_est_meteo['viento_velocidad']} de las {n_est_gold} estaciones del gold (la "
     f"fracción fina, en {n_est_meteo['fraccion_fina']}: falta donde PM10 es 0), así que sus medias mezclan estaciones. "
     "La anomalía dentro de la estación (el valor menos la media de su estación sobre las horas del gold) no depende de "
     "qué estaciones caen en cada régimen:", "",
     al.tabla_md(tabla_anom), "",
     "### Archivos", "",
     "- `reports/kmeans_propuesto.json` queda con la receta final " + ("(igual a `configs/kmeans.json`)" if propuesta_final == receta_final
     else "(**distinta** de `configs/kmeans.json`: revisar)") + " y reemplaza la propuesta automática del bloque «Receta propuesta» de arriba.",
     f"- `reports/centroides_sklearn.json`: scikit-learn {sklearn.__version__} con la receta final sobre todo el silver de Go "
     f"({al.miles(len(X_eq))} filas, la mejor de {len(SEMILLAS_REF)} semillas, hasta converger; inercia {in_eq:.4f})"
     + (f"; frente a Go, máx |Δ centroide| {DIF_GO[0]:.1e} tras emparejar e inercia relativa {DIF_GO[1]:.1e}." if DIF_GO else "."),
     "- `reports/descriptores_regimen.csv`: la tabla de descriptores con todas las columnas.", ""]
ruta_dec = REPORTS / "calibracion_decision.md"
previo = ruta_dec.read_text(encoding="utf-8") if ruta_dec.exists() else "# Calibración del K-Means — tabla de decisión\n"
previo = previo.split("\n## Decisión del equipo")[0].rstrip("\n") + "\n"
ruta_dec.write_text(previo + "\n" + "\n".join(S), encoding="utf-8")
print("Escrito", ruta_dec)
display(Markdown("\n".join(S)))